## **AGP SLIDES - UK BUSINESSES**

In [14]:
# Import packages and set filepaths
import pandas as pd
import numpy as np
import altair as alt
from pathlib import Path
from pandas.api.types import CategoricalDtype
import os
import eco_style 
alt.themes.enable("report")

script_dir = Path.cwd()
import_path = script_dir.parent / "Data"
chart_path = script_dir.parent / "Charts"

export_path = script_dir.parent / "Tables"


In [15]:
# IMPORT DATASETS
population_df = pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='population')
firm_dynamics_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='firm_dynamics')
job_flows_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='job_flows')
site_dynamics_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='site_dynamics')
cohort_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='cohort_analysis')
growth_rates_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='growth_rates')
growth_cats_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='growth_cats')
prod_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='prod')
sector_summary_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='sector_summary')


In [16]:
# Force data types to numeric where possible, * for diclosive values automatically read the column in as a string
def convert_numeric_columns(df):
    """
    Convert all numeric columns to numeric type, replacing '*' with NaN.
    Preserves non-numeric columns (like year, category names) as they are.
    """
    df_converted = df.copy()
    
    for col in df_converted.columns:
        # Try to convert to numeric, replacing '*' and other errors with NaN
        df_converted[col] = pd.to_numeric(df_converted[col].replace('*', np.nan), errors='ignore')
    
    return df_converted

# Apply to all dataframes
population_df = convert_numeric_columns(population_df)
firm_dynamics_df = convert_numeric_columns(firm_dynamics_df)
job_flows_df = convert_numeric_columns(job_flows_df)
site_dynamics_df = convert_numeric_columns(site_dynamics_df)
cohort_df = convert_numeric_columns(cohort_df)
growth_rates_df = convert_numeric_columns(growth_rates_df)
growth_cats_df = convert_numeric_columns(growth_cats_df)
prod_df = convert_numeric_columns(prod_df)

/var/folders/dp/dmj0l1fj4cld5dkvtqpy9bqc0000gn/T/ipykernel_38233/2220193345.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_converted[col] = pd.to_numeric(df_converted[col].replace('*', np.nan), errors='ignore')
/var/folders/dp/dmj0l1fj4cld5dkvtqpy9bqc0000gn/T/ipykernel_38233/2220193345.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_converted[col] = pd.to_numeric(df_converted[col].replace('*', np.nan), errors='ignore')


In [ ]:
# Economic contribution by firm size 2023

size_df = population_df[population_df['dimension']=='Size']
size_df_2023 = size_df[size_df['year']==2023]


size_df_2023['Total_employment'] = size_df_2023['employment'].sum()
size_df_2023['Total_firms'] = size_df_2023['n_firms'].sum()
size_df_2023['Total_turnover'] = size_df_2023['turnover'].sum()

size_df_2023['Share_of_employment'] = size_df_2023['employment']/size_df_2023['Total_employment']
size_df_2023['Share_of_firms'] = size_df_2023['n_firms']/size_df_2023['Total_firms']
size_df_2023['Share_of_turnover'] = size_df_2023['turnover']/size_df_2023['Total_turnover']

firmsize_share_of_activity = size_df_2023.melt(id_vars='category',
                                                value_vars=['Share_of_employment','Share_of_firms','Share_of_turnover'],
                                                value_name='Share of activity')

label_map = {
    'Share_of_employment': 'Employment',
    'Share_of_firms': 'Firms',
    'Share_of_turnover': 'Turnover'
}

firmsize_share_of_activity['variable'] = firmsize_share_of_activity['variable'].map(label_map)

sizeband_order = ['Micro (0-9)', 'Small (10-49)', 'Medium (50-249)', 'Large (250+)']
variable_order = ['Firms', 'Employment', 'Turnover']

chart = alt.Chart(firmsize_share_of_activity).mark_bar().encode(
    x=alt.X('category:O', sort=sizeband_order),
    y=alt.Y('Share of activity:Q', axis=alt.Axis(format='%'), scale=alt.Scale(domain=[0, 1])),
    color=alt.Color('variable:N', sort=variable_order).legend(title=None, orient='bottom', 
        direction='horizontal'),
    xOffset=alt.XOffset('variable:N', sort=variable_order)
)

display(chart)
#chart.save(chart_path / 'Descriptive paper/Composition/BSD_firmsize_contribution.png', scale_factor=2.0)

In [52]:
# Age composition of firms in 2023

age_population_df = population_df[population_df['dimension']=='Age']

age_df_2023 = age_population_df[age_population_df['year']==2023]



age_df_2023['Total_employment'] = age_df_2023['employment'].sum()
age_df_2023['Total_firms'] = age_df_2023['n_firms'].sum()
age_df_2023['Total_turnover'] = age_df_2023['turnover'].sum()

age_df_2023['Share_of_employment'] = age_df_2023['employment']/age_df_2023['Total_employment']
age_df_2023['Share_of_firms'] = age_df_2023['n_firms']/age_df_2023['Total_firms']
age_df_2023['Share_of_turnover'] = age_df_2023['turnover']/age_df_2023['Total_turnover']

firmsize_share_of_activity = age_df_2023.melt(id_vars='category',
                                                value_vars=['Share_of_employment','Share_of_firms','Share_of_turnover'],
                                                value_name='Share of activity')

label_map = {
    'Share_of_employment': 'Employment',
    'Share_of_firms': 'Firms',
    'Share_of_turnover': 'Turnover'
}

firmsize_share_of_activity['variable'] = firmsize_share_of_activity['variable'].map(label_map)

age_order = ['New (0-2 years)','Young (3-5 years)','Mature (6-10 years)','Old (11+ years)']
variable_order = ['Firms', 'Employment', 'Turnover']

chart = alt.Chart(firmsize_share_of_activity).mark_bar().encode(
    x=alt.X('category:O', sort=age_order),
    y=alt.Y('Share of activity:Q', axis=alt.Axis(format='%'), scale=alt.Scale(domain=[0, 1])),
    color=alt.Color('variable:N', sort=variable_order).legend(title=None, orient='bottom', 
        direction='horizontal'),
    xOffset=alt.XOffset('variable:N', sort=variable_order)
)

display(chart)
chart.save(chart_path / 'AGP Charts/BSD_firmsage_contribution.png', scale_factor=2.0)

/var/folders/dp/dmj0l1fj4cld5dkvtqpy9bqc0000gn/T/ipykernel_24620/3082372232.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  age_df_2023['Total_employment'] = age_df_2023['employment'].sum()
/var/folders/dp/dmj0l1fj4cld5dkvtqpy9bqc0000gn/T/ipykernel_24620/3082372232.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  age_df_2023['Total_firms'] = age_df_2023['n_firms'].sum()
/var/folders/dp/dmj0l1fj4cld5dkvtqpy9bqc0000gn/T/ipykernel_24620/3082372232.py:11: SettingWithCopyWarning: 
A value is trying to be

alt.Chart(...)

In [53]:

# Share of firms vs share of employment — grouped horizontal bars by sector

sector_population_df = population_df[population_df['dimension'] == 'Sector']
sector_population_df_2023 = sector_population_df[sector_population_df['year'] == 2023].copy()

total_firms = sector_population_df_2023["n_firms"].sum()
total_emp   = sector_population_df_2023["employment"].sum()
sector_population_df_2023["Share of firms (%)"]      = (sector_population_df_2023["n_firms"]     / total_firms * 100).round(2)
sector_population_df_2023["Share of employment (%)"] = (sector_population_df_2023["employment"]  / total_emp   * 100).round(2)

sector_order = sector_population_df_2023.sort_values("Share of firms (%)", ascending=False)["category"].tolist()

df_long = sector_population_df_2023.melt(
    id_vars="category",
    value_vars=["Share of firms (%)", "Share of employment (%)"],
    var_name="measure",
    value_name="share",
)

chart = alt.Chart(df_long).mark_bar().encode(
    x=alt.X("share:Q", title="Share of total (%)"),
    y=alt.Y("category:N", sort=sector_order, title=None, axis=alt.Axis(labelLimit=180)),
    yOffset=alt.YOffset("measure:N", sort=["Share of firms (%)", "Share of employment (%)"]),
    color=alt.Color(
        "measure:N",
        scale=alt.Scale(
            domain=["Share of firms (%)", "Share of employment (%)"],
            range=["#378add", "#e05b2b"],
        ),
        legend=alt.Legend(title=None, orient="right"),
    ),
    tooltip=[
        alt.Tooltip("category:N", title="Sector"),
        alt.Tooltip("measure:N",  title="Measure"),
        alt.Tooltip("share:Q",    title="Share (%)", format=".2f"),
    ],
).properties(
    width=560,
    height=440,
 #   title=alt.TitleParams(
 #       subtitle="Sectors sorted by share of firms",
 #       "Share of firms vs share of employment by sector (2023)",
 #       subtitleColor="#888",
 #       fontSize=15,
 #       subtitleFontSize=12,
 #   )
).configure_view(
    strokeWidth=0
).configure_axis(
    grid=True, gridColor="#eeeeee", labelFontSize=12, titleFontSize=13
)

chart.show()
chart.save(chart_path / 'AGP Charts/BSD_sector_composition_2023.png', scale_factor=2)


alt.Chart(...)

In [54]:
# CHANGE IN RELATIVE SHARE OF FIRMS AND EMPLOYMENT ACROSS SECTORS
START_YEAR = 2000
END_YEAR = 2023

# --- Filter to sector dimension and endpoint years ---
df_sector = population_df[
    (population_df['dimension'] == 'Sector') &
    (population_df['year'].isin([START_YEAR, END_YEAR]))
].copy()

# --- Compute shares for both measures ---
records = []
for measure, label in [('n_firms', 'Firm share'), ('employment', 'Employment share')]:
    temp = df_sector[['year', 'category', measure]].copy()
    totals = temp.groupby('year')[measure].sum().reset_index()
    totals.columns = ['year', 'total']
    temp = temp.merge(totals, on='year')
    temp['share'] = (temp[measure] / temp['total']) * 100
    pivoted = temp.pivot(index='category', columns='year', values='share').reset_index()
    pivoted.columns = ['category', 'start_share', 'end_share']
    pivoted['change'] = pivoted['end_share'] - pivoted['start_share']
    pivoted['measure'] = label
    records.append(pivoted[['category', 'change', 'measure']])

plot_df = pd.concat(records, ignore_index=True)

# --- Sort industries by firm share change ---
sort_order = (
    plot_df[plot_df['measure'] == 'Firm share']
    .sort_values('change')['category']
    .tolist()
)

# --- Grouped bar chart ---
bars = alt.Chart(plot_df).mark_bar().encode(
    x=alt.X('change:Q',
             axis=alt.Axis(title='Change in share (pp)', format='+.1f',
                           grid=True, gridOpacity=0.3),
             scale=alt.Scale(domain=[
                 plot_df['change'].min(),
                 plot_df['change'].max()
             ])),
    y=alt.Y('category:N',
             sort=sort_order,
             axis=alt.Axis(title=None, labelFontSize=11)),
    color=alt.Color('measure:N',
                     scale=alt.Scale(
                         domain=['Firm share','Employment share'],
                         range=["#179fdb","#e05b2b"]
                     ),
                     legend=alt.Legend(title=None, orient='none',
                                       legendX=420,   
                                       legendY=10,  
                                       direction='vertical')),
    yOffset=alt.YOffset('measure:N', sort=['Firm share', 'Employment share'])
)

# --- Zero line ---
rule = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(
    color='#374151', strokeWidth=1
).encode(x='x:Q')

# --- Combine ---
chart = (bars + rule).properties(
    width=500,
    height=400
).configure_view(
    strokeWidth=0
)

display(chart)
chart.save(chart_path / 'AGP Charts/BSD_sector_change.png', scale_factor=2.0)

alt.LayerChart(...)

In [23]:
# Productivity disparities by industry

sector_prod_df = prod_df[prod_df['dimension']=='Sector']
sector_prod_df_2023 = sector_prod_df[sector_prod_df['year']==2023]

sector_prod_df_2023 = sector_prod_df_2023.sort_values("p50_prod", ascending=False)
sector_order = sector_prod_df_2023["category"].tolist()

y = alt.Y("category:N", sort=sector_order, title=None, axis=alt.Axis(labelLimit=180))

tooltip = [
    alt.Tooltip("category:N",    title="Sector"),
    alt.Tooltip("p10_prod:Q",  title="P10",    format=","),
    alt.Tooltip("p25_prod:Q",  title="P25",    format=","),
    alt.Tooltip("p50_prod:Q",  title="Median", format=","),
    alt.Tooltip("p75_prod:Q",  title="P75",    format=","),
    alt.Tooltip("p90_prod:Q",  title="P90",    format=","),
    alt.Tooltip("mean_prod:Q", title="Mean",   format=","),
]

whisker = alt.Chart(sector_prod_df_2023).mark_rule(color="#b5d4f4", strokeWidth=2).encode(
    x=alt.X("p10_prod:Q", title="Turnover per employee (£ thousand)", axis=alt.Axis(format="~s")),
    x2="p90_prod:Q",
    y=y,
    tooltip=tooltip,
)

iqr_box = alt.Chart(sector_prod_df_2023).mark_bar(color="#378add", height=14).encode(
    x=alt.X("p25_prod:Q"),
    x2="p75_prod:Q",
    y=y,
    tooltip=tooltip,
)

median_tick = alt.Chart(sector_prod_df_2023).mark_tick(
    color="white", thickness=2, size=14
).encode(
    x=alt.X("p50_prod:Q"),
    y=y,
    tooltip=tooltip,
)

mean_circle = alt.Chart(sector_prod_df_2023).mark_point(
    shape="circle", filled=True, color="white", size=40,
    stroke="#378add", strokeWidth=1.5
).encode(
    x=alt.X("mean_prod:Q"),
    y=y,
    tooltip=tooltip,
)

chart = (whisker + iqr_box + median_tick + mean_circle).properties(
    width=620,
    height=480,
#    title=alt.TitleParams(
#        "Productivity Distribution by Sector (2023)",
#        subtitle="Line shows p10–p90 range · Boxes show IQR (p25–p75) · White tick = median · Circle = mean",
#        subtitleColor="#888",
#        subtitleFontSize=12,
#        fontSize=15,
#    )
).configure_view(
    strokeWidth=0
).configure_axis(
    grid=True,
    gridColor="#eeeeee",
    labelFontSize=12,
    titleFontSize=13,
)

chart.show()
chart.save(chart_path / 'AGP Charts/RLP_by_sector.png', scale_factor=2)

alt.LayerChart(...)

In [57]:
# EASE OF DOING BUSINESS

business_ease_df = pd.read_excel(import_path / 'Policy/Starting a Business.xlsx')

business_ease_df

,Unnamed: 0,Location,Starting a Business score,Procedure – Men (number),Time – Men (days),Cost – Men (% of income per capita),Procedure – Women (number),Time – Women (days),Cost – Women (% of income per capita),Paid-in min. capital (% of income per capita),Starting a Business rank
0,Region,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,East Asia & Pacific,83.9,6.5,25.6,17.4,6.6,25.7,17.4,3.5,99
2,NaN,Europe & Central Asia,90.5,5.2,11.9,4.0,5.2,11.9,4.0,0.7,57
3,NaN,Latin America & Caribbean,79.6,8.1,28.8,31.4,8.1,28.8,31.4,0.4,119
4,NaN,Middle East & North Africa,84.0,6.5,19.7,16.7,7.1,20.3,16.7,8.9,105
...,...,...,...,...,...,...,...,...,...,...,...
217,NaN,Vietnam,85.1,8.0,16.0,5.6,8.0,16.0,5.6,0.0,115
218,NaN,West Bank and Gaza,70.2,10.0,43.0,40.3,11.0,44.0,40.3,0.0,173
219,NaN,"Yemen, Rep.",76.8,6.0,40.0,40.2,7.0,41.0,40.2,0.0,156
220,NaN,Zambia,84.9,7.0,8.5,34.0,7.0,8.5,34.0,0.0,117


In [ ]:
# Starting a Business score — G7 countries

g7_names = {
    'Canada': 'Canada',
    'France': 'France',
    'Germany': 'Germany',
    'Italy': 'Italy',
    'Japan': 'Japan',
    'United Kingdom': 'United Kingdom',
    'United States': 'United States',
}

g7_df = business_ease_df[
    business_ease_df['Location'].isin(g7_names.keys())
][['Location', 'Starting a Business score']].copy()

g7_df['Starting a Business score'] = pd.to_numeric(g7_df['Starting a Business score'], errors='coerce')
g7_df = g7_df.dropna(subset=['Starting a Business score'])

sort_order = g7_df.sort_values('Starting a Business score', ascending=False)['Location'].tolist()

chart = alt.Chart(g7_df).mark_bar().encode(
    x=alt.X('Starting a Business score:Q',
            scale=alt.Scale(domain=[0, 100]),
            axis=alt.Axis(title='Starting a Business score')),
    y=alt.Y('Location:N', sort=sort_order, title=None),
    color=alt.condition(
        alt.datum['Location'] == 'United Kingdom',
        alt.value('#e05b2b'),
        alt.value('#378add'),
    ),
    tooltip=[
        alt.Tooltip('Location:N', title='Country'),
        alt.Tooltip('Starting a Business score:Q', title='Score', format='.1f'),
    ],
).properties(
    width=500,
    height=280,
)

display(chart)
# chart.save(chart_path / 'AGP Charts/starting_a_business_g7.png', scale_factor=2.0)


In [69]:
# Starting a Business score — G20 countries
# Note: World Bank naming conventions used (e.g. "Korea, Rep.", "Turkiye")

#https://archive.doingbusiness.org/en/data/exploretopics/starting-a-business

g20_names = [
    'Argentina', 'Australia', 'Brazil', 'Canada', 'China',
    'France', 'Germany', 'India', 'Indonesia', 'Italy',
    'Japan', 'Korea, Rep.', 'Mexico', 'Russian Federation',
    'Saudi Arabia', 'South Africa', 'Turkiye',
    'United Kingdom', 'United States',
]

g20_df = business_ease_df[
    business_ease_df['Location'].isin(g20_names)
][['Location', 'Starting a Business score']].copy()

g20_df['Starting a Business score'] = pd.to_numeric(g20_df['Starting a Business score'], errors='coerce')
g20_df = g20_df.dropna(subset=['Starting a Business score'])

sort_order = g20_df.sort_values('Starting a Business score', ascending=False)['Location'].tolist()

chart = alt.Chart(g20_df).mark_bar().encode(
    x=alt.X('Starting a Business score:Q',
            scale=alt.Scale(domain=[0, 100]),
            axis=alt.Axis(title='Starting a Business score')),
    y=alt.Y('Location:N', sort=sort_order, title=None),
    color=alt.condition(
        alt.datum['Location'] == 'United Kingdom',
        alt.value('#e05b2b'),
        alt.value('#378add'),
    ),
    tooltip=[
        alt.Tooltip('Location:N', title='Country'),
        alt.Tooltip('Starting a business score:Q', title='Score', format='.1f'),
    ],
).properties(
    title='Regulatory burden in starting a business across G20 countries',
    width=500,
    height=400,
)

display(chart)
chart.save(chart_path / 'AGP Charts/starting_a_business_g20.png', scale_factor=2.0)


alt.Chart(...)

In [ ]:
# Employment protection strictness — G20 countries (OECD EPL data)

epl_df = pd.read_csv(import_path / 'Policy/OECD_employment_protection.csv')

# G20 country names as they appear in the OECD data
g20_epl_names = [
    'Australia', 'Canada', 'France', 'Germany', 'India',
    'Indonesia', 'Italy', 'Japan', 'Korea', 'Mexico',
    'Saudi Arabia', 'South Africa', 'Türkiye',
    'United Kingdom', 'United States',
    # Argentina, Brazil, China, Russia not typically covered by OECD EPL
]

# Filter: G20 countries, individual dismissals (regular contracts), most recent year
epl_g20 = epl_df[
    (epl_df['Reference area'].isin(g20_epl_names)) &
    (epl_df['MEASURE'] == 'EPL_R')
].copy()

epl_g20['OBS_VALUE'] = pd.to_numeric(epl_g20['OBS_VALUE'], errors='coerce')
epl_g20['TIME_PERIOD'] = pd.to_numeric(epl_g20['TIME_PERIOD'], errors='coerce')

# Take most recent non-null observation per country
latest_year = (
    epl_g20.dropna(subset=['OBS_VALUE'])
    .groupby('Reference area')['TIME_PERIOD']
    .max()
    .reset_index()
    .rename(columns={'TIME_PERIOD': 'latest_year'})
)
epl_g20 = epl_g20.merge(latest_year, on='Reference area')
epl_g20 = epl_g20[epl_g20['TIME_PERIOD'] == epl_g20['latest_year']].dropna(subset=['OBS_VALUE'])

sort_order = epl_g20.sort_values('OBS_VALUE', ascending=False)['Reference area'].tolist()

chart = alt.Chart(epl_g20).mark_bar().encode(
    x=alt.X('OBS_VALUE:Q',
            scale=alt.Scale(domain=[0, 6]),
            axis=alt.Axis(title='Employment protection strictness (0–6 scale)')),
    y=alt.Y('Reference area:N', sort=sort_order, title=None),
    color=alt.condition(
        alt.datum['Reference area'] == 'United Kingdom',
        alt.value('#e05b2b'),
        alt.value('#378add'),
    ),
    tooltip=[
        alt.Tooltip('Reference area:N', title='Country'),
        alt.Tooltip('OBS_VALUE:Q', title='EPL score', format='.2f'),
        alt.Tooltip('TIME_PERIOD:Q', title='Year', format='d'),
    ],
).properties(
    width=500,
    height=400,
)

display(chart)
# chart.save(chart_path / 'AGP Charts/employment_protection_g20.png', scale_factor=2.0)


In [66]:
# Costs of restructuring per employee, given in months of employee compensation. Dta is based on rstructuring plans from 191 large listed companies.

ranks_data = {
    'Country': ['Switzerland', 'Denmark', 'United States', 'Sweden', 'United Kingdom',
                'Germany', 'Netherlands', 'France', 'Italy', 'Spain'],
    'Rank': [3, 3, 7, 10, 18, 31, 31, 38, 52, 62],
}
ranks_df = pd.DataFrame(ranks_data)

sort_order = ranks_df.sort_values('Rank', ascending=False)['Country'].tolist()

chart = alt.Chart(ranks_df).mark_bar().encode(
    x=alt.X('Rank:Q', axis=alt.Axis(title='Cost of restructuring per employee (months of compensation)')),
    y=alt.Y('Country:N', sort=sort_order, title=None),
    color=alt.condition(
        alt.datum['Country'] == 'United Kingdom',
        alt.value('#e05b2b'),
        alt.value('#378add'),
    ),
    tooltip=[
        alt.Tooltip('Country:N', title='Country'),
        alt.Tooltip('Rank:Q', title='Rank'),
    ],
).properties(
    title='Restructuring costs per employee, across countries',
    width=500,
    height=320,
)

display(chart)
chart.save(chart_path / 'AGP Charts/restructuring_costs_coste_costanlem_2025.png', scale_factor=2.0)


alt.Chart(...)

In [17]:
vat_counts

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,average 85-95,2610.2,728.5,6096.3,Unnamed: 12,linear decline,235,34.8,160,Unnamed: 17
0,NaN,SUM:,171194,34373,243169,168549,33501.0,236745,NaN,4539.2,705.3,3364.4,NaN,stalled businesses,12192.0,2236.8,11866.0,26294.8
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,missing businesses,9547.0,1364.8,5442.0,NaN
2,Band,Turnover,Sole Traders,Partnerships,Companies,STs extrap,Pships extrap,Cs extrap,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,£66-67k,8648,1325,9222,8648,1325,9222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,£67-68k,8568,1304,8947,8568,1304,8947,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,3,£68-69k,8356,1367,9189,8356,1367,9189,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,4,£69-70k,8208,1406,9014,8208,1406,9014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,5,£70-71k,8325,1439,9482,8325,1439,9482,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,6,£71-72k,7587,1324,9094,7587,1324,9094,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,7,£72-73k,7461,1317,9185,7461,1317,9185,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
vat_counts_long

,Turnover,variable,value
0,£10-11k,Sole Traders,114234
1,£11-12k,Sole Traders,109589
2,£12-13k,Sole Traders,109628
3,£13-14k,Sole Traders,97925
4,£14-15k,Sole Traders,96557
...,...,...,...
415,£145-146k,Companies,3001
416,£146-147k,Companies,2978
417,£147-148k,Companies,2942
418,£148-149k,Companies,2854


In [26]:
# VAT Threshold

vat_counts = pd.read_excel(import_path / 'Policy/VAT_2018-19.xlsx', sheet_name='66-96k',skiprows=6)

vat_counts_long = vat_counts.melt(id_vars='Turnover',value_vars=['Sole Traders','Partnerships','Companies'])

turnover_order = [
    '£66-67k', '£67-68k', '£68-69k', '£69-70k', '£70-71k', '£71-72k',
    '£72-73k', '£73-74k', '£74-75k', '£75-76k', '£76-77k', '£77-78k',
    '£78-79k', '£79-80k', '£80-81k', '£81-82k', '£82-83k', '£83-84k',
    '£84-85k', '£85-86k', '£86-87k', '£87-88k', '£88-89k', '£89-90k',
    '£90-91k', '£91-92k', '£92-93k', '£93-94k', '£94-95k', '£95-96k',
]

colour_scale = alt.Scale(
    domain=['Sole Traders', 'Partnerships', 'Companies'],
    range=[eco_style.pallete['nominal_1'], eco_style.pallete['nominal_2'], eco_style.pallete['nominal_3']],
)

x_enc = alt.X('Turnover:O', sort=turnover_order,
               axis=alt.Axis(values=turnover_order[::4], labelAngle=0, title='Turnover band'))
colour_enc = alt.Color('variable:N', scale=colour_scale, legend=None)

lines = alt.Chart(vat_counts_long).mark_line().encode(
    x=x_enc,
    y=alt.Y('value:Q'),
    color=colour_enc,
)

end_labels = alt.Chart(vat_counts_long[vat_counts_long['Turnover'] == turnover_order[-1]]).mark_text(
    align='left', dx=6, fontSize=11
).encode(
    x=x_enc,
    y=alt.Y('value:Q'),
    text=alt.Text('variable:N'),
    color=colour_enc,
)

vat_rule = alt.Chart(pd.DataFrame({'Turnover': ['£85-86k']})).mark_rule(
    strokeDash=[4, 4], color=eco_style.pallete['domain'], strokeWidth=1.5, opacity=0.7
).encode(
    x=alt.X('Turnover:O', sort=turnover_order),
)

chart = (lines + vat_rule + end_labels).properties(title="Bunching of firms under VAT threshold (2018/19)")
chart.save(chart_path / 'AGP Charts/VAT_threshold_201819.png', scale_factor=2.0)